# PackingSolver exploration

the visualisation bit needs plotly and also nbformat>=4.2.0

In [1]:
import json
import sys
from importlib.metadata import version
from pathlib import Path

import packingsolver.rectangle as psr

print("Python:", sys.version)
print("PackingSolver:", version("packingsolver"))

Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
PackingSolver: 0.1.1340


Load one existing dataset

In [2]:
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

dataset_path = project_root / "data" / "synthetic" / "pilot_v1.json"

with dataset_path.open() as file:
    data = json.load(file)

datasets = data["datasets"]

In [3]:
dataset = next(
    dataset
    for dataset in datasets
    if dataset["dataset_id"]
    == "rect_v1_mixed_sheet2500x1250_n35_seed0"
)

parts = dataset["parts"]
sheet_w, sheet_h = dataset["sheet_size"]

print("Dataset:", dataset["dataset_id"])
print("Parts:", len(parts))
print("Sheet size:", (sheet_w, sheet_h))

Dataset: rect_v1_mixed_sheet2500x1250_n35_seed0
Parts: 35
Sheet size: (2500, 1250)


### Build the instance

- Using the plain bin-packing objective: place all requested parts while minimising the number of sheets. 
- PackingSolver distinguishes this from BinPackingWithLeftovers, which additionally considers the leftover on the last sheet. We should leave that extra objective out of the initial comparison.
- rotation not yet established

In [4]:
available_sheets = len(parts)

builder = psr.InstanceBuilder()
builder.set_objective(psr.Objective.BinPacking)

builder.add_bin_type(
    sheet_w,
    sheet_h,
    copies=available_sheets,
)

for part_id, width, height in parts:
    builder.add_item_type(
        width,
        height,
        copies=1,
    )

instance = builder.build()

### Run the solver

In [5]:
parameters = psr.OptimizeParameters()
parameters.time_limit = 5

output = psr.optimize(instance, parameters)

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  35
Number of items:       35
Number of bin types:   1
Number of bins:        35
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       12133094
Mean item width:       827.143
Mean item height:      420.943
Smallest item width:   139
Smallest item height:  139
Total item profit:     1.21331e+07
Largest item profit:   745240
Total item weight:     0
Largest item copies:   1
Total bin area:        109375000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
       0.160       5           22.35                  TS g 0 d X q 1

Final statistics
----------------
Time (s):              5.00589
Is proven infeasible:  0
Primal bound:          5
Dual bound:    

### Inspect and visualise the result

In [6]:
print(output)
print(output.solution)

Time (s):              5.00589
Is proven infeasible:  0
Primal bound:          5
Dual bound:            4
Gap:                   20%



In [7]:
psr.visualize(output.solution).show()

***
- Enable rotation
- The mapping preserves original part identities instead of relying on dimensions to identify parts. This matters when multiple parts have identical dimensions.

In [4]:
rotation = True
available_sheets = len(parts)

builder = psr.InstanceBuilder()
builder.set_objective(psr.Objective.BinPacking)

builder.add_bin_type(
    sheet_w,
    sheet_h,
    copies=available_sheets,
)

item_type_to_part_id = {}

for part_id, width, height in parts:
    item_type_id = builder.add_item_type(
        width,
        height,
        copies=1,
        oriented=not rotation,
    )

    item_type_to_part_id[item_type_id] = part_id

instance = builder.build()

Optimisiation cell

In [5]:
parameters = psr.OptimizeParameters()
parameters.time_limit = 5

output = psr.optimize(instance, parameters)

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  35
Number of items:       35
Number of bin types:   1
Number of bins:        35
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       12133094
Mean item width:       827.143
Mean item height:      420.943
Smallest item width:   139
Smallest item height:  139
Total item profit:     1.21331e+07
Largest item profit:   745240
Total item weight:     0
Largest item copies:   1
Total bin area:        109375000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
       0.390       5           22.35                  TS g 1 d Y q 1

Final statistics
----------------
Time (s):              5.00526
Is proven infeasible:  0
Primal bound:          5
Dual bound:    

Export and inspect solution
- For ITEM rows, LX and LY are the dimensions after placement, so they already account for rotation. BIN identifies the sheet containing the item

In [6]:
import pandas as pd

results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

solution_path = results_dir / "pilot_solution.csv"

output.solution.write(str(solution_path))

solution_df = pd.read_csv(solution_path)

display(solution_df.head(10))
print("Columns:", solution_df.columns.tolist())

,TYPE,ID,COPIES,BIN,X,Y,LX,LY,GROUP_ID
0,BIN,0,1,0,0,0,2500,1250,NaN
1,ITEM,28,1,0,0,0,1307,500,0.0
2,ITEM,18,1,0,1307,0,601,1240,0.0
3,ITEM,29,1,0,0,500,1268,341,0.0
4,ITEM,2,1,0,1908,0,588,821,0.0
5,ITEM,21,1,0,0,841,1306,308,0.0
6,ITEM,12,1,0,1908,821,508,391,0.0
7,BIN,0,1,1,0,0,2500,1250,NaN
8,ITEM,15,1,1,0,0,1362,327,0.0
9,ITEM,1,1,1,1362,0,597,1247,0.0


Columns: ['TYPE', 'ID', 'COPIES', 'BIN', 'X', 'Y', 'LX', 'LY', 'GROUP_ID']


Convert to shared layout format

This produces precisely the representation the RectPack adapter returns: (bin_id, x, y, width, height, part_id)

In [7]:
bin_rows = (
    solution_df.loc[solution_df["TYPE"] == "BIN"]
    .sort_values("BIN")
)

item_rows = solution_df.loc[solution_df["TYPE"] == "ITEM"]

if bin_rows.empty or item_rows.empty:
    raise ValueError(
        "No non-empty packing was exported; inspect the solver output."
    )

if not bin_rows["COPIES"].eq(1).all():
    raise ValueError(
        "This initial converter expects one physical sheet per BIN row."
    )

bin_id_map = {
    int(original_bin_id): new_bin_id
    for new_bin_id, original_bin_id in enumerate(bin_rows["BIN"])
}

used_sheets = [
    (int(row.LX), int(row.LY))
    for row in bin_rows.itertuples(index=False)
]

placements = [
    (
        bin_id_map[int(row.BIN)],
        int(row.X),
        int(row.Y),
        int(row.LX),
        int(row.LY),
        item_type_to_part_id[int(row.ID)],
    )
    for row in item_rows.itertuples(index=False)
]

print("Used sheets:", len(used_sheets))
print("Placed parts:", len(placements))
print("First placements:", placements[:5])

Used sheets: 5
Placed parts: 35
First placements: [(0, 0, 0, 1307, 500, 'P00028'), (0, 1307, 0, 601, 1240, 'P00018'), (0, 0, 500, 1268, 341, 'P00029'), (0, 1908, 0, 588, 821, 'P00002'), (0, 0, 841, 1306, 308, 'P00021')]


Run shared metrics

In [8]:
from cutting_benchmark.rectangle_metrics import (
    calculate_rectangle_metrics,
)

metrics = calculate_rectangle_metrics(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
)

display(pd.Series(metrics, name="PackingSolver"))

parts_total                      35
parts_packed                     35
complete                       True
individually_feasible          True
sheets_used                       5
requested_area             12133094
packed_area                12133094
part_fraction                   1.0
area_fraction                   1.0
utilisation                0.776518
area_lower_bound                  4
sheet_gap_to_area_bound           1
Name: PackingSolver, dtype: object

check identity preservation explicitly:

In [9]:
requested_ids = {part_id for part_id, _, _ in parts}
placed_ids = [placement[5] for placement in placements]

assert len(placed_ids) == len(set(placed_ids)), "Duplicate part placements"
assert set(placed_ids) <= requested_ids, "Unknown part IDs"

missing_ids = requested_ids - set(placed_ids)

print("Missing parts:", sorted(missing_ids))

Missing parts: []


Layout validation

In [10]:
from cutting_benchmark.rectangle_validation import (
    validate_rectangle_layout,
)

validation = validate_rectangle_layout(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
    available_sheets=available_sheets,
)

if not validation["layout_valid"]:
    raise ValueError(
        "Invalid PackingSolver layout:\n"
        + "\n".join(validation["validation_errors"])
    )

metrics = calculate_rectangle_metrics(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
)

display(pd.Series({
    "layout_valid": validation["layout_valid"],
    **metrics,
}))

layout_valid                   True
parts_total                      35
parts_packed                     35
complete                       True
individually_feasible          True
sheets_used                       5
requested_area             12133094
packed_area                12133094
part_fraction                   1.0
area_fraction                   1.0
utilisation                0.776518
area_lower_bound                  4
sheet_gap_to_area_bound           1
dtype: object

*** 
Try the packing solver function from the cutting_benchmark package

In [11]:
from cutting_benchmark.solvers.packingsolver_adapter import (
    run_pack as run_packingsolver,
)

result = run_packingsolver(
    parts=dataset["parts"],
    sheet_size=dataset["sheet_size"],
    rotation=True,
    time_limit=5,
    return_layout=True,
)

display(pd.Series({
    key: value
    for key, value in result.items()
    if key not in {"placements", "used_sheets"}
}))

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  35
Number of items:       35
Number of bin types:   1
Number of bins:        35
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       12133094
Mean item width:       827.143
Mean item height:      420.943
Smallest item width:   139
Smallest item height:  139
Total item profit:     1.21331e+07
Largest item profit:   745240
Total item weight:     0
Largest item copies:   1
Total bin area:        109375000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
       0.340       5           22.35                  TS g 1 d Y q 1

Final statistics
----------------
Time (s):              5.00491
Is proven infeasible:  0
Primal bound:          5
Dual bound:    

solver                          packingsolver
config_id                  BinPacking__time5s
objective                          BinPacking
rotation                                 True
time_limit_seconds                          5
sheet_width                              2500
sheet_height                             1250
available_sheets                           35
runtime_seconds                      5.006043
layout_valid                             True
parts_total                                35
parts_packed                               35
complete                                 True
individually_feasible                    True
sheets_used                                 5
requested_area                       12133094
packed_area                          12133094
part_fraction                             1.0
area_fraction                             1.0
utilisation                          0.776518
area_lower_bound                            4
sheet_gap_to_area_bound           

In [12]:
display(
    pd.DataFrame(
        result["placements"],
        columns=[
            "bin_id",
            "x",
            "y",
            "width",
            "height",
            "part_id",
        ],
    ).head()
)

,bin_id,x,y,width,height,part_id
0,0,0,0,1307,500,P00028
1,0,1307,0,601,1240,P00018
2,0,0,500,1268,341,P00029
3,0,1908,0,588,821,P00002
4,0,0,841,1306,308,P00021


Run a small benchmark

In [13]:
from cutting_benchmark.benchmark import run_benchmark

configurations = [
    {"rotation": False, "time_limit": 5},
    {"rotation": True, "time_limit": 5},
]

benchmark_results = run_benchmark(
    datasets=[dataset],
    configurations=configurations,
    solver=run_packingsolver,
    repeats=1,
)

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  35
Number of items:       35
Number of bin types:   1
Number of bins:        35
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       12133094
Mean item width:       827.143
Mean item height:      420.943
Smallest item width:   139
Smallest item height:  139
Total item profit:     1.21331e+07
Largest item profit:   745240
Total item weight:     0
Largest item copies:   1
Total bin area:        109375000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
       0.316       5           22.35                  TS g 1 d Y q 1

Final statistics
----------------
Time (s):              5.02937
Is proven infeasible:  0
Primal bound:          5
Dual bound:    

In [14]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_results.to_csv(
    results_dir / "packingsolver_pilot_raw.csv",
    index=False,
)